In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/TwinCheck'
REPO = '/content/Group-Project'
OUT = f'{DRIVE}/results/F10'
F5OUT = f'{DRIVE}/results/F5'
CKPT = f'{DRIVE}/models/F5'
CKPT10 = f'{DRIVE}/models/F10'
CACHE = f'{DRIVE}/data/processed/vision/genimage_debiased96_jpeg.pkl'
!rm -rf {REPO}
!git clone -q --depth 1 https://github.com/himaallu/Group-Project.git {REPO}
!unzip -o -q {DRIVE}/F10/f10_code.zip -d {REPO}
!pip -q install onnx onnxruntime
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
import sys

import pandas as pd
import torch

os.chdir(REPO)
sys.path.insert(0, REPO)
os.makedirs(OUT, exist_ok=True)
os.makedirs(CKPT10, exist_ok=True)
twins = pd.read_csv('data/copies/vision/verite_vae_twins.csv')
needed = [CACHE, f'{CKPT}/resnet50_s42.pt', f'{CKPT}/vit_b16_s42.pt', f'{F5OUT}/train_resnet50_s42.json', f'{F5OUT}/train_vit_b16_s42.json',
          f'{DRIVE}/features/vision/genimage/clip_vitl14_debiased96/embeddings_normalized.npy',
          f'{DRIVE}/features/vision/genimage/clip_vitl14_debiased96/embedding_manifest.csv',
          f'{DRIVE}/features/vision/verite/siglip2_so400m/embeddings_normalized.npy',
          f'{DRIVE}/features/vision/verite/siglip2_text/embeddings_normalized.npy',
          f'{DRIVE}/features/vision/verite/siglip2_vae_twins/embeddings_normalized.npy',
          *[f'{DRIVE}/{p}' for p in twins['absolute_path']], *[f'{DRIVE}/{p}' for p in twins['twin_path']]]
missing = [p for p in needed if not os.path.exists(p)]
print('files needed:', len(needed), '| missing:', len(missing))
for p in missing[:20]:
    print('  missing:', p)
assert not missing
assert os.path.exists('src/vision/leakage_cv.py'), 'f10_code.zip was not unpacked'
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
SMOKE = '/content/smoke'
!rm -rf {SMOKE}
!python -m pytest -q tests/vision/test_f10.py
for arch, extra in (('resnet50', ''), ('vit_b16', ''), ('resnet50', '--freeze-bn')):
    !python -m src.vision.train_detector --arch {arch} --seed 43 {extra} --cache {CACHE} --limit 5 --epochs 1 --batch 4 --ckpt-dir {SMOKE}/ckpt --out {SMOKE}/out --preds-dir {SMOKE}/out
!python -m src.vision.quantize_static --ckpt {SMOKE}/ckpt/resnet50_s43.pt --cache {CACHE} --work {SMOKE}/onnx --out {SMOKE}/out --limit 5 --calib 10 --check 10 --warmup 1 --runs 3
print(sorted(os.listdir(f'{SMOKE}/out')))

In [ ]:
import json

ref = pd.read_csv('results/F5/cnn_vit.csv')
ref = ref[ref['corruption'] == 'clean']
rows = []
for arch in ('resnet50', 'vit_b16'):
    !python -m src.vision.train_detector --arch {arch} --seed 42 --eval-only --ckpt {CKPT}/{arch}_s42.pt --cache {CACHE} --out {OUT} --preds-dir {OUT}
    run = json.load(open(f'{OUT}/eval_{arch}_s42_check.json'))
    for split in ('validation', 'test'):
        r = ref[(ref['model'] == arch) & (ref['split'] == split)].iloc[0]
        rows.append({'model': arch, 'split': split, **{m: abs(run[split][m] - r[m]) for m in ('accuracy', 'precision', 'recall', 'f1', 'auroc')}})
check = pd.DataFrame(rows)
check.to_csv(f'{OUT}/rescore_check.csv', index=False)
print(check.to_string(index=False))
assert check[['accuracy', 'precision', 'recall', 'f1', 'auroc']].to_numpy().max() < 1e-3, 'Re-scored F5 checkpoints do not match cnn_vit.csv: stop here'
print('The cached input reproduces F5.')

In [ ]:
import hashlib
import io
from concurrent.futures import ThreadPoolExecutor

import numpy as np
from PIL import Image, ImageOps, JpegImagePlugin
from sklearn.linear_model import LogisticRegression
from torchvision import transforms
from transformers import CLIPVisionModelWithProjection

SIZE, POOL_SIZE, SEED, BATCH = 96, 500, 42, 128
CLIP_DIR = f'{DRIVE}/features/vision/verite/clip_vitl14_debiased96'
gen = pd.concat([pd.read_csv(f'data/processed/vision/genimage_{s}_manifest.csv') for s in ('train', 'validation', 'test')], ignore_index=True)


def jpeg_settings(path):
    with Image.open(path) as im:
        return [list(t) for t in im.quantization.values()], JpegImagePlugin.get_sampling(im)


def load_square(path):
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im).convert('RGB')
        w, h = im.size
        s = min(w, h)
        left, top = (w - s) // 2, (h - s) // 2
        return np.asarray(im.crop((left, top, left + s, top + s)).resize((SIZE, SIZE), Image.Resampling.BICUBIC))


def encode(a, key):
    tables, sampling = JPEG_POOL[int(hashlib.md5(key.encode()).hexdigest()[:8], 16) % len(JPEG_POOL)]
    buf = io.BytesIO()
    Image.fromarray(a).save(buf, format='JPEG', qtables=tables, subsampling=sampling)
    return buf.getvalue()


def process(paths):
    paths = list(paths)
    with ThreadPoolExecutor(32) as pool:
        arrays = list(pool.map(lambda p: load_square(f'{DRIVE}/{p}'), paths))
    with ThreadPoolExecutor(8) as pool:
        return list(pool.map(encode, arrays, paths))


if not os.path.exists(f'{CLIP_DIR}/twins.npy'):
    pool_rows = gen[(gen['split'] == 'train') & (gen['class_name'] == 'real') & (gen['original_mode'] == 'RGB')].sample(POOL_SIZE, random_state=SEED)
    with ThreadPoolExecutor(32) as pool:
        settings = list(pool.map(lambda p: jpeg_settings(f'{DRIVE}/{p}'), pool_rows['raw_export_path']))
    JPEG_POOL = [s for s in settings if len(s[0]) >= 2 and s[1] in (0, 1, 2)]
    print('JPEG settings from real training photos:', len(JPEG_POOL))
    clip = CLIPVisionModelWithProjection.from_pretrained('openai/clip-vit-large-patch14', torch_dtype=torch.float16).to('cuda').eval()
    to_tensor = transforms.Compose([
        transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711]),
    ])

    @torch.inference_mode()
    def embed(blobs):
        out = []
        for i in range(0, len(blobs), BATCH):
            x = torch.stack([to_tensor(Image.open(io.BytesIO(b)).convert('RGB')) for b in blobs[i:i + BATCH]])
            e = clip(pixel_values=x.to('cuda', clip.dtype)).image_embeds.float()
            out.append(torch.nn.functional.normalize(e, dim=1).cpu().numpy())
        return np.concatenate(out)

    os.makedirs(CLIP_DIR, exist_ok=True)
    np.save(f'{CLIP_DIR}/originals.npy', embed(process(twins['absolute_path'])))
    np.save(f'{CLIP_DIR}/twins.npy', embed(process(twins['twin_path'])))
    pd.DataFrame({'image_file': twins['original_filename']}).to_csv(f'{CLIP_DIR}/manifest.csv', index=False)
    del clip
    torch.cuda.empty_cache()

feat = f'{DRIVE}/features/vision/genimage/clip_vitl14_debiased96'
X = np.load(f'{feat}/embeddings_normalized.npy').astype(np.float32)
fm = pd.read_csv(f'{feat}/embedding_manifest.csv')
train = (fm['split'] == 'train').to_numpy()
probe = LogisticRegression(max_iter=2000, random_state=SEED).fit(X[train], (fm.loc[train, 'class_name'] == 'fake').astype(int))
f3_auth = pd.read_csv('results/F3/F3_verite_authenticity_scores.csv')
new = {v: probe.predict_proba(np.load(f'{CLIP_DIR}/{v}.npy'))[:, 1] for v in ('originals', 'twins')}
clip_check = pd.DataFrame([{'version': v, 'photos': len(new[v]),
                            'max_abs_diff_vs_F3': float(np.abs(new[v] - f3_auth[c].to_numpy()).max()),
                            'same_flag_vs_F3': float(((new[v] >= 0.5) == (f3_auth[c].to_numpy() >= 0.5)).mean())}
                           for v, c in (('originals', 'original_authenticity_score'), ('twins', 'twin_authenticity_score'))])
clip_check.to_csv(f'{OUT}/verite_clip_check.csv', index=False)
print(clip_check.to_string(index=False))

In [ ]:
!python -m src.vision.leakage_cv --drive {DRIVE} --out {OUT}

In [ ]:
!python -m src.vision.train_detector --arch resnet50 --seed 42 --eval-only --restore-imagenet-bn --ckpt {CKPT}/resnet50_s42.pt --cache {CACHE} --out {OUT} --preds-dir {OUT}

In [ ]:
import datetime
import subprocess

UAE = datetime.timezone(datetime.timedelta(hours=4))
EPOCHS = 3
RUNS = [('resnet50', 43, CKPT, F5OUT, []), ('vit_b16', 43, CKPT, F5OUT, []),
        ('resnet50', 44, CKPT, F5OUT, []), ('vit_b16', 44, CKPT, F5OUT, []),
        ('resnet50', 42, CKPT10, OUT, ['--freeze-bn'])]
epoch_seconds = {}


def seconds_left(k, epochs_done):
    left = 0.0
    for j, (arch, *_) in enumerate(RUNS[k:]):
        n = EPOCHS - epochs_done if j == 0 else EPOCHS
        left += n * epoch_seconds.get(arch, max(epoch_seconds.values())) * 1.1
    return left


for k, (arch, seed, ckpt_dir, out_dir, extra) in enumerate(RUNS):
    name = f'{arch}_s{seed}' + ('_freezebn' if extra else '')
    if os.path.exists(f'{out_dir}/train_{name}.json'):
        print('already done:', name)
        continue
    cmd = [sys.executable, '-m', 'src.vision.train_detector', '--arch', arch, '--seed', str(seed), '--cache', CACHE,
           '--ckpt-dir', ckpt_dir, '--out', out_dir, '--preds-dir', OUT, *extra]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='')
        if '"seconds"' in line:
            h = json.loads(line)
            epoch_seconds[arch] = h['seconds']
            eta = datetime.datetime.now(UAE) + datetime.timedelta(seconds=seconds_left(k, h['epoch']))
            print(f'>>> {name} epoch {h["epoch"]} took {h["seconds"] / 60:.1f} min. All five runs should finish around {eta:%H:%M} UAE time.', flush=True)
    assert proc.wait() == 0, name

In [ ]:
!python -m src.vision.f10_stats --f5 {F5OUT} --f6 results/F6 --out {OUT}

In [ ]:
!python -m src.vision.quantize_static --ckpt {CKPT}/resnet50_s42.pt --cache {CACHE} --work {CKPT} --out {OUT}
if os.path.exists(f'{CKPT10}/resnet50_s42_freezebn.pt'):
    !python -m src.vision.quantize_static --ckpt {CKPT10}/resnet50_s42_freezebn.pt --cache {CACHE} --work {CKPT10} --out {OUT}

In [ ]:
import shutil

from google.colab import files

PACK = '/content/F10_results'
shutil.rmtree(PACK, ignore_errors=True)
shutil.copytree(OUT, f'{PACK}/results/F10')
os.makedirs(f'{PACK}/results/F5', exist_ok=True)
for arch in ('resnet50', 'vit_b16'):
    for seed in (43, 44):
        p = f'{F5OUT}/train_{arch}_s{seed}.json'
        if os.path.exists(p):
            shutil.copy(p, f'{PACK}/results/F5/')
shutil.make_archive(PACK, 'zip', PACK)
print(sorted(os.listdir(f'{PACK}/results/F10')))
print(sorted(os.listdir(f'{PACK}/results/F5')))
files.download(f'{PACK}.zip')